<h1><center>Workshop on Computer Vision for Image Segmentation</center></h1>

Prepared by **Vladimir PIMONOV**  
Affiliation: [ILM / Team PNEC]  
Contact: [foxlightmanstudes@gmail.com]

These notebooks were prepared for the workshop and are intended as guided practical material.  
They can also be used independently outside the workshop.  
For questions, feedback, or bug reports, please contact the author.

<h2><center>PW 04. Image inference</center></h2>

In this notebook, we use a trained segmentation model for **inference** on microscopy images.

The goal is no longer to optimize the model, but to apply it to new data and inspect the predicted results.

The main operations are:

1. load the trained model weights,
2. select one or more input images,
3. run inference either on full images or on individual patches,
4. visualize the predicted probability maps and binary masks,
5. overlay contours on the original image,
6. optionally apply a simple post-processing step to suppress weak isolated false detections.

This notebook focuses on the practical use of a trained model.  
It is therefore centered on prediction, thresholding, visualization, and interpretation of the outputs.

## 1. Import the required libraries

We begin by importing the packages needed for model loading, image reading, prediction, and visualization.

These imports combine three types of tools:

- general-purpose Python and scientific libraries,
- PyTorch components used to load and run the network,
- the workshop utility functions for inference and post-processing.

In particular, this notebook uses the inference helpers that were defined in the project library:

- patch inference,
- tiled full-image inference,
- and confidence-based filtering of weak predictions. 

In [ ]:
import os
from os.path import join as pjoin
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

import matplotlib.pyplot as plt

import tifffile
from tifffile import TiffFile

import numpy as np
import torch
import torch.utils.data

from torchvision.transforms import functional as TF

### Select correct progressbar widget depending on the system used for environment
if "VSCODE_PID" in os.environ:
    from tqdm import tqdm
else:
    from tqdm.notebook import tqdm

import gc
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

# personalized library for recognition and training
from scripts.engine import *
from scripts.engine import _batch_to_tensor, _batch_to_tensor_mc
from scripts.loss_functions import *
from scripts.transforms import *

# Library of models and inferences (Deep versions of UNet and UNet++ are legacy versions that have been used for training)
from scripts.models.UNetPPResNetDeep_lagacy import UNetPPResNetDeep as UNetPPResNetDeep_l
from scripts.models.UNetResNetDeep_legacy import UNetResNetDeep as UNetResNetDeep_l
from scripts.models.UNet3PlusResNet34 import UNet3PlusResNet34
from scripts.models.UNet3PlusResNet50Deep import UNet3PlusResNet50Deep
from scripts.models.UNetPPResNet34 import UNetPPResNet34
from scripts.models.UNetResNet34 import UNetResNet34
from scripts.inference import infer_full_image_tiled, infer_patch, filter_mask_by_confident_overlap

## 2. Check whether GPU acceleration is available and use it

In [ ]:
### Check if cuda (gpu acceleration is avaliable)
torch.cuda.is_available()

In [ ]:
### Use gpu acceleration if possible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 3. Inference Model Selection

---

### 3.1. Overview

There are **6 models from 3 U-Net families** available for inference.

**U-Net** - the original encoder-decoder segmentation architecture introduced by Ronneberger, Fischer, and Brox in *U-Net: Convolutional Networks for Biomedical Image Segmentation* ([paper](https://arxiv.org/abs/1505.04597)). It is built from a contracting path and an expanding path connected by same-scale skip connections, which helps combine semantic context with spatial detail.

**U-Net++** - a nested variant of U-Net introduced in *UNet++: A Nested U-Net Architecture for Medical Image Segmentation* ([paper](https://arxiv.org/abs/1807.10165)). Compared with the original U-Net, it replaces the direct skip connections with dense, intermediate convolutional pathways, which helps reduce the semantic gap between encoder and decoder features.

**U-Net 3+** - a full-scale skip-connection variant introduced in *UNet 3+: A Full-Scale Connected UNet for Medical Image Segmentation* ([paper](https://arxiv.org/abs/2004.08790)). In contrast to U-Net and U-Net++, it aggregates features from all encoder and decoder scales at each decoder stage, improving multi-scale context integration and boundary awareness.

For each of these three model families, two implementations are available:

- **Classic** version: a shallower implementation with **5 convolutional levels**.
- **Deep** version: an extended implementation with **7 convolutional levels**, adding extra downsampling stages to improve large-context perception and strengthen the model's ability to capture coarse, global structures in large images.

Overall, the available inference models are:

1. **U-Net Classic** Model size: ~114 MB 
2. **U-Net Deep** Model size: ~408 MB (also available in legacy format with larger deep conv layers)
3. **U-Net++ Classic** Model size: ~102 MB
4. **U-Net++ Deep** Model size: ~625 MB (also available in legacy format with larger deep conv layers)
5. **U-Net 3+ Classic** Model size: ~88 MB
6. **U-Net 3+ Deep** Model size: ~222 MB

### 3.2. Architectural variants

Each family is available in two architectural implementations:

- **Classic**: a shallower implementation with **5 convolutional levels**
- **Deep**: an extended implementation with **7 convolutional levels**, including additional downsampling stages to improve large-context perception and strengthen coarse-scale feature extraction

### 3.3. Main differences

| Variant | Depth | Main characteristic | Typical benefit |
|---|---:|---|---|
| Classic | 5 levels | Standard encoder-decoder depth | Lower memory use, faster inference |
| Deep | 7 levels | Additional encoder depth and larger receptive field | Better large-context understanding and coarse structure detection |

Each model use ResNet architectures as backbones and can use already trained ResNets (34 and 50) which can help to speed up convergence.

## 4. Trained versions

Each architecture is usually available in **three trained versions**, corresponding to different training losses.

### Loss-based versions

| Version | Loss function |
|---|---|
| vN.0 | `[Binary cross entropy + Dice]` |
| vN.1 | `[Binary cross entropy]` |
| vN.2 | `[Focal loss]` |

Example:
- `v1.0` = BCE + Dice
- `v1.1` = BCEWithLogitsLoss
- `v1.3` = Focal loss

Each model trained for 200 epochs and can have several checkpoints. The smalles epochs number checkpoint corresponds to the early stop training (where validation set loss stopped improving, deduced from training log plots) the last checkpoint corresponds to the epoch 201 (the last epoch). There might be intermediate epoch saved to test.

---

## Models

**NB!** For details of training hyperparameters refer to the files in the folder of the checkpoint modes "Readme.txt" for the trainin logs use "history vXX.X.json" files for each version of model.

---

### 1. U-Net Classic
- **Architecture:** 5 convolutional levels, backbone ResNet-34

#### Checkpoints
- **v24** (4 fully trained models)

  **Path**:  
  Models\U-Net (resnet34 5conv layers) LR v24 512pix CapAE dataset
  
|Version|Loss|Available checkpoints|File name|
|---|---|---|---|
|V24.0|BCE + Dice|61, 201|U-Net (resnet34) [checkpoint number] epoch v24.0|
|V24.1|Weighted BCE|81, 201|U-Net (resnet34) [checkpoint number] epoch v24.1|
|V24.2|Focal loss|81, 201|U-Net (resnet34) [checkpoint number] epoch v24.2|
|V24.3|Focal loss (optimized)|81, 201|U-Net (resnet34) [checkpoint number] epoch v24.3|

---

### 2. U-Net Deep
- **Architecture:** 7 convolutional levels, backbone ResNet-50 or ResNet-34, ResNet-50 legacy model in use for inference

#### Checkpoints
- **v25** (3 fully trained models)

  **Path**:  
  Models\U-Net deep (resnet50 7conv layers) LR v25 512pix CapAE dataset
  
|Version|Loss|Available checkpoints|File name|
|---|---|---|---|
|V25.0|BCE + Dice|81, 201|U-Net (resnet50) [checkpoint number] epoch BCE+Dice v25.0|
|V25.1|Weighted BCE|81, 201|U-Net (resnet50) [checkpoint number] epoch BCE weight v25.1|
|V25.2|Focal loss|61, 201|U-Net (resnet50) [checkpoint number] epoch Focal loss v25.2|

---

### 3. U-Net++ Classic 
- **Architecture:** 5 convolutional levels, backbone ResNet-34

#### Checkpoints
- **v23** (3 fully trained models)

  **Path**:  
  Models\U-Net++ (resnet34 5conv layers) LR v23 512pix CapAE dataset
  
|Version|Loss|Available checkpoints|File name|
|---|---|---|---|
|V23.0|BCE + Dice|61, 201|U-Net++ (resnet34) [checkpoint number] epoch BCE + Dice v23.0|
|V23.1|Weighted BCE|61, 201|U-Net++ (resnet34) [checkpoint number] epoch BCE weight v23.1|
|V23.2|Focal loss|61, 201|U-Net++ (resnet34) [checkpoint number] epoch focal loss v23.2|

---

### 4. U-Net++ Deep 
- **Architecture:** 7 convolutional levels, backbone ResNet-34 or ResNet-50, ResNet-50 legacy model in use for inference

#### Checkpoints
- **v26** (4 fully trained models)

  **Path**:  
  Models\U-Net++ deep (resnet50 7conv layers) LR v26 512pix CapAE dataset
  
|Version|Loss|Available checkpoints|File name|
|---|---|---|---|
|V26.0|BCE + Dice|81, 201|U-Net++ (resnet50 deep) [checkpoint number] epoch v26.0|
|V26.1|Weighted BCE|101, 201|U-Net++ (resnet50 deep) [checkpoint number] epoch v26.1|
|V26.2|Focal loss|61, 201|U-Net++ (resnet50 deep) [checkpoint number] epoch v26.2|
|V26.3|Focal loss (optimized)|61, 201|U-Net++ (resnet50 deep) [checkpoint number] epoch v26.3|

---

### 5. U-Net3+ Classic 
- **Architecture:** 5 convolutional levels, backbone ResNet-34 or ResNet-50, ResNet-50 legacy model in use for inference

#### Checkpoints
- **v30** (3 fully trained models)

  **Path**:  
  Models\U-Net3+ (resnet34 5conv layers) LR v30 512pix CapAE dataset
  
|Version|Loss|Available checkpoints|File name|
|---|---|---|---|
|V30.0|BCE + Dice|61, 201|U-Net3+ (resnet34) [checkpoint number] epoch v30.0|
|V30.1|Weighted BCE|81, 201|U-Net3+ (resnet34) [checkpoint number] epoch v30.1|
|V30.3|Focal loss (optimized)|61, 201|U-Net3+ (resnet34) [checkpoint number] epoch v30.3|

---

### 6. U-Net3+ Deep 
- **Architecture:** 7 convolutional levels, backbone ResNet-34 or ResNet-50, ResNet-50 legacy model in use for inference

#### Checkpoints
- **v29** (3 fully trained models)

  **Path**:  
  Models\U-Net3+ deep (resnet50 7conv layers) LR v29 512pix CapAE dataset
  
|Version|Loss|Available checkpoints|File name|
|---|---|---|---|
|V29.0|BCE + Dice|81, 201|U-Net3+ (resnet50) [checkpoint number] epoch v29.0|
|V29.1|Weighted BCE|81, 201|U-Net3+ (resnet50) [checkpoint number] epoch v29.1|
|V29.3|Focal loss (optimized)|61, 201|U-Net3+ (resnet50) [checkpoint number] epoch v29.3|



In [ ]:
### Select model for the inference

### 1. U-Net classic
model = UNetResNet34(num_classes=1, pretrained_backbone=True, in_channels=1).to(device)

### 2. U-Net Deep
### (available with resnet 50 and resnet34, on inference available the resnet50 version)

# model = UNetResNetDeep_l(backbone="resnet34", in_channels=1, num_classes=1, pretrained=True).to(device)
# model = UNetResNetDeep_l(backbone="resnet50", in_channels=1, num_classes=1, pretrained=True).to(device)

### 3. U-Net++ classic
# model = UNetPPResNet34(num_classes=1, pretrained_backbone=True, in_channels=1).to(device)

### 4. U-Net++ Deep
# model = UNetPPResNetDeep_l(backbone="resnet50", in_channels=1, num_classes=1, pretrained=True,
#                         extra_down_channels=(1024, 1024),
#                         dec_channels=(64, 128, 256, 512, 512, 512, 512)).to(device)

### 5. U-Net3+ classic
# model = UNet3PlusResNet34(in_channels=1, num_classes=1, pretrained=True, fuse_ch=64).to(device)

### 6. U-Net3+ Deep
# model = UNet3PlusResNet50Deep(in_channels=1, num_classes=1, pretrained=True, fuse_ch=64).to(device)

## 5. Load the trained weights

At this stage, the architecture has already been selected.  
Now we load the corresponding trained weights from a saved checkpoint.

This step transfers the learned parameters into the chosen model structure.

A critical point is that the checkpoint must match the model architecture exactly.  
The shape and organization of the learned weights depend on the network definition, so a checkpoint trained with one architecture cannot be loaded into another one.

If the architecture and checkpoint are consistent, PyTorch reports that all keys were matched successfully.

In [ ]:
### select the model path and model checkpoint to pass to the following step
### NB! The model and the checpoint should be of the same type otherwise the weights won't be uploade

model_path = pjoin('Models', 'U-Net (resnet34 5conv layers) LR v24 512pix CapAE dataset')
model_title = 'U-Net (resnet34) 201 epoch v24.3'

model.load_state_dict(torch.load(os.path.join(model_path,
                            f'{model_title}.pth'),
                            map_location=device))

### if everything have been made correctly below will appear message
### All keys matched successfully

## 6. Select the input images and prepare full-image inference

This section applies the trained model to one or several larger microscopy images stored in a folder.

Unlike the training notebooks, where the model worked mainly on prepared patches, here the goal is to process real images from disk and save the resulting predictions.

We first define the directory containing the images to analyse and collect the TIFF files that will be passed to the model.

The filtering conditions in the file search allow us to keep only the relevant files.  
For example, the code excludes mask files and can also restrict the selection to a specific subset of images depending on the folder structure or naming convention.

This step therefore builds the list of images on which inference will be performed.

The output of the model will be generated in two forms:

- a **probability map**, which contains continuous confidence values between 0 and 1,
- a **binary prediction map**, obtained by thresholding the probability map. :contentReference[oaicite:0]{index=0}

The probability map preserves more information and is useful for analysis and visualization.  
The binary map is more directly usable when a final segmentation mask is needed.

In [ ]:
### Use files from the batch that haven't been used at all in any step of training

root = pjoin('Agregates for training TP', 'Co150V_C_A1_TM22-36')

images = [os.path.join(r,file) for r,d,f in os.walk(root)
          for file in f
          if file.lower().endswith('.tiff') and 'mask' not in file.lower()
          and 'Co150V' in r and 'Inference' not in r
          ### magnification below 200k
#           and int(file.split('-')[2][:-1]) < 200001
         ]

print("Total %.i images ready for inference" % len(images))

## 7. Run tiled inference and save the outputs

Large microscopy images are often too large to process in a single forward pass.  
For that reason, this notebook uses **tiled inference**: the image is split into overlapping patches, each patch is processed independently, and the predicted probabilities are then merged back into a full-size map. Overlapping predictions are averaged, which usually makes the reconstruction smoother and more stable near patch borders. :contentReference[oaicite:2]{index=2}

A few parameters are especially important here:

- **`patch_size`** defines the size of the tiles given to the model,
- **`step`** defines the distance between consecutive tiles,
- **`thr`** defines the probability threshold used to convert the probability map into a binary mask. :contentReference[oaicite:3]{index=3}

### 8.1 Meaning of the threshold

The network outputs a probability-like confidence map after applying the sigmoid function to the raw logits.  
The threshold then decides which pixels are considered foreground.

- A **lower threshold** makes the prediction more permissive.  
  This usually increases recall, but may also create more false positives.

- A **higher threshold** makes the prediction more conservative.  
  This usually reduces false positives, but may miss weak or uncertain regions.

There is no universal best threshold.  
Its value should be chosen according to the task and the acceptable balance between missed detections and false alarms.

### 8.2 Why save both probability and binary maps?

Both outputs are useful for different reasons.

- The **binary mask** is a direct segmentation result.
- The **probability map** keeps the model confidence and can later be re-thresholded, visualized, or post-processed.

The code saves both maps as 8-bit TIFF images so they can be opened easily in tools such as ImageJ.

In [ ]:
### Use the full tiled image inference file to make the full image

pro_thr = 0.4

for img_path in tqdm(images[:]):
    
    ### Isolate the image title
    file_path, file_name_with_extention = os.path.split(img_path)
    file_name, file_extention = os.path.splitext(file_name_with_extention)
    
    ### read the file
    with TiffFile(img_path) as tif:
        img = tif.asarray()
        
    ### Make full inference use threshold of 0.4 for prediction
    prob_map, pred_map = infer_full_image_tiled(
        model.to(device),
        img_np=img,   # HxW
        patch_size=512,
        step=512/2,
        device=next(model.parameters()).device,
        thr=pro_thr,
        progress=tqdm
    )
    torch.cuda.empty_cache()
    
    model_spath = os.path.basename(model_path).split(' LR ')[0]
    recogn_vers = model_title.split(' ')[-1]
    
    saving_address = os.path.join(file_path, f'Inference {model_title}')
    if not os.path.exists(saving_address):
        os.makedirs(saving_address)
    else:
        pass
    
    ### Save the pretiction and probability in a form of 8 bit Tiff by multiplyin to 255
    ### To make it usable with ImageJ (it does not work with float tiffs)
    tifffile.imwrite(os.path.join(saving_address,
                           f"{file_name}_Bin_thr{pro_thr}_{recogn_vers}.tiff"),
                     np.uint8(pred_map*255))
    
    tifffile.imwrite(os.path.join(saving_address,
                           f"{file_name}_Prob_{recogn_vers}.tiff"),
                     np.uint8(prob_map*255))

## 8. Release unused GPU memory

After inference, some GPU memory may still be occupied by tensors that are no longer needed.

Clearing the CUDA cache does not change the prediction result itself, but it can help keep memory usage under control, especially when processing many large images one after another.

In [ ]:
# model.to("cpu")
torch.cuda.empty_cache()

## 9. Prepare and load one image for detailed inspection

The earlier part of the notebook is designed for **batch processing**, where many files are analysed automatically and the outputs are saved to disk.

This section is written for the complementary use case: checking the files **one by one**.

We first rebuild the list of candidate TIFF images available in the chosen folder, then select one image explicitly and load it as a NumPy array.

This type of workflow is useful when we want to:

- inspect a specific image in more detail,
- test parameters interactively,
- compare visual outputs more carefully,
- or verify how the model behaves on individual cases before launching or after completing batch inference.

In [ ]:
with TiffFile(images[8]) as tif:
    img = tif.asarray()

In [ ]:
img.shape

## 11. Compare a selected image patch with its predicted probability map

This section is intended for detailed local inspection of the model output on one selected patch.

Working at the patch level makes the prediction easier to interpret because we can focus on a smaller region without the added complexity of full-image reconstruction.

The function used here returns the **probability map** predicted by the model for the selected patch.

Displaying the original patch together with its probability map helps in several ways:

- it shows where the model places the most confident foreground regions,
- it reveals how sharp or diffuse the prediction is,
- it makes it easier to inspect uncertain boundaries or weak responses.

A smooth probability distribution is normal: the model usually expresses stronger confidence near the predicted core of an object and more uncertainty near its borders.

In [ ]:
### select patch size

inf_patch = 512

prob, pred = infer_patch(model.to(device), img[500:500+inf_patch,500:500+inf_patch], device)

plt.figure(figsize = (15, 10))
plt.imshow(img[500:500+inf_patch,500:500+inf_patch], cmap = 'gray')
plt.axis('off')
plt.show()

plt.figure(figsize = (15, 10))
plt.imshow(prob)
plt.axis('off')
plt.show()

## 12. Run full tiled inference on the selected image

This cell applies the same tiled-inference procedure as before, but now only on one chosen image.

This makes it easier to inspect:

- the total inference time,
- the effect of the chosen patch size,
- the effect of the chosen stride,
- and the final probability and binary maps.

The threshold used here again determines how the continuous confidence map is converted into a binary foreground mask.

In [ ]:
%%time

patch_size = 512
overlap = 1/2

### make full inference
prob_map, pred_map = infer_full_image_tiled(
    model.to(device),
    img_np=img,   # HxW
    patch_size=patch_size,
    step=patch_size*overlap,
    device=next(model.parameters()).device,
    thr=0.4,
    progress=tqdm
)
torch.cuda.empty_cache()

## 13. Visualize the binary prediction map

Here the probability map is converted into a binary mask using a fixed threshold and displayed on its own.

This representation is simpler than the probability map because every pixel is forced into one of two classes:

- foreground,
- background.

This view is useful when a final segmentation mask is needed, but it hides the confidence information contained in the original probability map.

In [ ]:
### Plot full image and inference

plt.figure(figsize = (15, 10))
plt.imshow(img, cmap = 'gray')
plt.axis('off')
plt.show()

plt.figure(figsize = (15, 10))
plt.imshow(prob_map)
plt.axis('off')
plt.show()

## 14. Overlay the thresholded contour on the image

After viewing the full image together with its prediction, it is often useful to superimpose the segmentation result directly on the original image.

One simple way to do this is to extract the contour of the **thresholded prediction mask** and draw it on top of the grayscale image.

This type of visualization is helpful because it preserves the original image content while making the predicted object boundaries easy to see.  
In many cases, it is easier to interpret the quality of a segmentation from a contour overlay than from a mask shown separately.

At this stage, the threshold plays an important role: it determines which probability values are considered foreground before the contour is extracted.

In [ ]:
### Plot full image and inference

plt.figure(figsize = (15, 10))
plt.imshow(img, cmap = 'gray')
plt.contour(prob_map>0.4, levels=[0.5], colors="red", linewidths=0.5)
plt.axis('off')
plt.show()


## 15. Compare several contour levels corresponding to different confidence thresholds

Choosing a threshold is not always straightforward.

A segmentation model produces a continuous probability map, but converting that map into a final binary mask requires selecting one threshold value.  
That choice affects the apparent size and shape of the detected regions.

To better understand this effect, it can be useful to display several contours corresponding to different confidence levels on the same image.

In this case, the contours are drawn directly from the **probability map** rather than from a single binary mask.  
This allows us to compare regions of different confidence on the same image.

Each contour level corresponds to a different confidence threshold.  
For example:

- a **low contour level** marks regions where the model already considers the presence of an object plausible,
- an **intermediate level** marks more confident regions,
- a **high level** marks only the most confident parts of the prediction.

This type of visualization is useful because it shows not only where the model predicts an object, but also **how strongly** it supports that prediction.

It can also help with threshold selection, which is often a non-trivial task.  
Checking several thresholds in this way can be a practical aid when choosing the most suitable operating point for a given application.

In [ ]:
### Plot full image and inference

plt.figure(figsize = (15, 10))
plt.imshow(img, cmap = 'gray')
plt.contour(prob_map, levels=[0.05, 0.5, 0.99], colors=["lightcoral", "red", "yellow"], linewidths=0.5)
plt.axis('off')
plt.show()


## 16. Filter weak isolated predictions using a high-confidence anchor

This section applies a simple post-processing rule to suppress weak isolated detections.

The idea is to use **two thresholds** instead of one:

- a **low threshold**, used to define all candidate predicted regions,
- a **high threshold**, used to define only the most confident cores.

Then the low-threshold mask is split into connected components, and only those components that overlap at least one high-confidence region are kept. Weak components that do not touch any confident core are removed.

### Meaning of the two thresholds

The two thresholds play different roles:

- The **low threshold** is permissive and helps recover the full extent of plausible objects, including uncertain borders.
- The **high threshold** is more selective and anchors the prediction to regions identified with strong confidence by the model.

This is useful for filtering out **hallucinations** and weak isolated false detections, which often appear only at lower confidence levels and do not contain a stable high-confidence core.

In practice, the high threshold must be greater than or equal to the low threshold.

If the low threshold is too low, many weak noisy regions may appear.  
If it is too high, real object borders may be lost.

If the high threshold is too low, weak false detections may still be validated.  
If it is too high, even real objects may fail to contain a strong enough anchor and may be removed.

### Meaning of connectivity

Connected components depend on how pixel neighbourhood is defined.

- With **4-connectivity** (connectivity = 1), only horizontal and vertical neighbours are connected.
- With **8-connectivity** (connectivity = 2), diagonal neighbours are also connected.

For thin or slanted structures, 8-connectivity is often more natural.

In [ ]:
### Filter the low cetrainty hallucinations by high cetrainty map

plt.figure(figsize = (15, 10))
plt.imshow(img, cmap = 'gray')
plt.contour(filter_mask_by_confident_overlap(prob_map, 0.2, 0.75, connectivity = 2),
            levels=[0.5], colors=["red"], linewidths=0.5)
plt.axis('off')
plt.show()


## What we achieved in this notebook

In this notebook, we used a trained segmentation model for practical inference on microscopy images.

More precisely, we:

- loaded a trained checkpoint,
- applied inference to images from disk,
- generated probability maps and binary prediction masks,
- visualized the predictions on full images and on selected regions,
- overlaid thresholded contours on the original images,
- compared several confidence thresholds,
- and used low- and high-threshold logic to filter weak isolated detections.

This notebook completes the workflow by showing how a trained model can be used in practice and how its outputs can be interpreted, visualized, and refined after prediction.